# LJ EOS targeted production runner

Самодостаточный notebook для запуска отдельных заранее заданных точек `(T, rho)` Lennard-Jones EOS. Он нужен для уточнения положения границ спинодали: результаты сразу дописываются в CSV, а уже выполненные точки пропускаются при повторном запуске.

Есть два режима задания задач: `TARGET_POINTS + SEEDS` запускает каждую точку по всем seed, а `TARGET_RUNS = [(T, rho, seed), ...]` запускает ровно указанные отдельные прогоны.

Для Kaggle с двумя GPU можно включить `USE_MULTIGPU_WORKERS = True`: notebook запустит независимый worker на каждую GPU и будет писать отдельные shard-файлы `eos_targeted_points_gpu<N>.csv`. Notebook не строит графики, не делает fit, не сохраняет траектории и не создаёт блочные CSV.


In [ ]:
# Установка и импорт зависимостей.
# В Colab перед запуском выберите Runtime -> Change runtime type -> GPU.

import os, sys, math, time, csv, subprocess, traceback, multiprocessing as mp
from pathlib import Path
from datetime import datetime, timezone
from statistics import mean, stdev

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'openmm[cuda12]', 'numpy', 'pandas', 'tqdm'])

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import openmm as mm
import openmm.app as app
from openmm import unit

print('Python:', sys.version.split()[0])
print('OpenMM:', mm.version.version)
print('\n===== nvidia-smi =====')
try:
    smi = subprocess.run(['nvidia-smi'], text=True, capture_output=True)
    print(smi.stdout if smi.stdout else smi.stderr)
except FileNotFoundError:
    print('nvidia-smi not found')

OPENMM_PLATFORMS = [mm.Platform.getPlatform(i).getName() for i in range(mm.Platform.getNumPlatforms())]
print('OpenMM platforms:', OPENMM_PLATFORMS)
if 'CUDA' not in OPENMM_PLATFORMS:
    raise RuntimeError('CUDA is not available in OpenMM. Stop here: this production notebook is intended for Colab GPU runs, not CPU.')


In [ ]:
# Google Drive и директория вывода.

USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = Path('/content/drive/MyDrive/VPV_LJ/data')
else:
    OUTPUT_DIR = Path('/content/VPV_LJ/data')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUTPUT_DIR:', OUTPUT_DIR)


In [ ]:
# Конфигурация targeted production-run.

N = 2048

# Режим 1: каждая пара (T, rho) запускается для каждого seed из SEEDS.
TARGET_POINTS = [
    (1.30, 0.010),
    (1.30, 0.015),
    (1.30, 0.020),
    (1.30, 0.040),
    (1.30, 0.050),
    (1.30, 0.080),
    (1.30, 0.120),
    (1.30, 0.160),
    (1.30, 0.200),
    (1.30, 0.300),
    (1.30, 0.400),
    (1.30, 0.500),
    (1.30, 0.600),
    (1.30, 0.700),
    (1.30, 0.800),
    (1.40, 0.010),
    (1.40, 0.015),
    (1.40, 0.020),
    (1.40, 0.040),
    (1.40, 0.050),
    (1.40, 0.080),
    (1.40, 0.120),
    (1.40, 0.160),
    (1.40, 0.200),
    (1.40, 0.300),
    (1.40, 0.400),
    (1.40, 0.500),
    (1.40, 0.600),
    (1.40, 0.700),
    (1.40, 0.800),
    (1.50, 0.010),
    (1.50, 0.015),
    (1.50, 0.020),
    (1.50, 0.040),
    (1.50, 0.050),
    (1.50, 0.080),
    (1.50, 0.120),
    (1.50, 0.160),
    (1.50, 0.200),
    (1.50, 0.300),
    (1.50, 0.400),
    (1.50, 0.500),
    (1.50, 0.600),
    (1.50, 0.700),
    (1.50, 0.800),
]
SEEDS = [1, 2, 3, 4, 5]

# Режим 2: если список не пустой, запускаются только эти конкретные (T, rho, seed),
# а TARGET_POINTS и SEEDS игнорируются. Удобно для дозакрытия дырок в покрытии.
TARGET_RUNS = []

RUN_PREFIX = 'targeted_lj'

SIGMA = 1.0
EPSILON = 1.0
MASS = 1.0
RCUT = 2.5
DT = 0.002
FRICTION = 1.0

EQUIL_STEPS = 20_000
PROD_STEPS = 20_000
SAMPLE_INTERVAL = 500
N_BLOCKS = 10

# Kaggle обычно даёт две GPU с индексами 0 и 1. Если доступна только одна, notebook сам уйдёт в последовательный режим.
USE_MULTIGPU_WORKERS = True
REQUESTED_DEVICE_INDICES = ['0', '1']
DEVICE_INDEX = '0'
PRECISION = 'mixed'

def detect_cuda_device_indices():
    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=index', '--format=csv,noheader'],
            text=True, capture_output=True, check=False,
        )
        indices = [line.strip() for line in result.stdout.splitlines() if line.strip() != '']
        return indices
    except Exception:
        return []

AVAILABLE_DEVICE_INDICES = detect_cuda_device_indices()
DEVICE_INDICES = [str(i) for i in REQUESTED_DEVICE_INDICES if str(i) in AVAILABLE_DEVICE_INDICES]
if not DEVICE_INDICES:
    DEVICE_INDICES = [str(DEVICE_INDEX)]
USE_MULTIGPU_WORKERS = bool(USE_MULTIGPU_WORKERS and len(DEVICE_INDICES) > 1)
DEVICE_INDEX = DEVICE_INDICES[0]

print('Available CUDA device indices:', AVAILABLE_DEVICE_INDICES)
print('Selected CUDA device indices:', DEVICE_INDICES)
print('Use multi-GPU workers:', USE_MULTIGPU_WORKERS)

TARGET_POINTS = [(float(T), float(rho)) for T, rho in TARGET_POINTS]
TARGET_RUNS = [(float(T), float(rho), int(seed)) for T, rho, seed in TARGET_RUNS]
if len(set(TARGET_POINTS)) != len(TARGET_POINTS):
    raise ValueError('TARGET_POINTS contains duplicate (T, rho) pairs')
if len(set(TARGET_RUNS)) != len(TARGET_RUNS):
    raise ValueError('TARGET_RUNS contains duplicate (T, rho, seed) triples')

if TARGET_RUNS:
    REQUESTED_RUNS = TARGET_RUNS
    print('Mode: explicit TARGET_RUNS')
else:
    REQUESTED_RUNS = [(T, rho, int(seed)) for seed in SEEDS for T, rho in TARGET_POINTS]
    print('Mode: TARGET_POINTS x SEEDS')

print('Requested runs:', len(REQUESTED_RUNS))
print('Unique target points:', len(set((T, rho) for T, rho, seed in REQUESTED_RUNS)))
print('Seeds:', sorted(set(seed for T, rho, seed in REQUESTED_RUNS)))


In [ ]:
# Минимальные функции targeted production-runner.

EOS_FIELDS = ['run_id','seed','run_seed','T_target','rho_target','N','box_length','sigma','epsilon','mass','rcut','dt','friction','equil_steps','prod_steps','sample_interval','n_samples','n_blocks','platform','device_index','precision','wall_time_s','steps_per_second','T_mean','P_mean','U_mean','K_mean','E_mean','T_std','P_std','U_std','K_std','E_std','status']
FAILURE_FIELDS = ['run_id','seed','run_seed','T_target','rho_target','error_type','error_message','timestamp']

def run_id_for(seed, T, rho):
    text = RUN_PREFIX + '_seed' + str(int(seed)) + '_T' + format(float(T), '.3f') + '_rho' + format(float(rho), '.4f')
    return text.replace('.', 'p')

def deterministic_run_seed(seed, T, rho):
    # Stable integer seed derived from the physical point, independent of list order.
    t_code = int(round(float(T) * 1000.0))
    rho_code = int(round(float(rho) * 10000.0))
    return int(seed) * 1000000 + t_code * 1000 + rho_code + 17

def initialize_positions(N, L):
    n_side = math.ceil(N ** (1.0 / 3.0))
    spacing = L / n_side
    coords = []
    for ix in range(n_side):
        for iy in range(n_side):
            for iz in range(n_side):
                if len(coords) == N:
                    return np.asarray(coords, dtype=float)
                coords.append([(ix + 0.5) * spacing, (iy + 0.5) * spacing, (iz + 0.5) * spacing])
    return np.asarray(coords, dtype=float)

def create_lj_system(N, rho):
    L = float((int(N) / float(rho)) ** (1.0 / 3.0))
    system = mm.System()
    for _ in range(int(N)):
        system.addParticle(MASS * unit.dalton)
    system.setDefaultPeriodicBoxVectors(unit.Quantity(mm.Vec3(L,0,0), unit.nanometer), unit.Quantity(mm.Vec3(0,L,0), unit.nanometer), unit.Quantity(mm.Vec3(0,0,L), unit.nanometer))
    expr = '4*epsilon*((sigma/r)^12-(sigma/r)^6)-4*epsilon*((sigma/rcut)^12-(sigma/rcut)^6)'
    force = mm.CustomNonbondedForce(expr)
    force.addGlobalParameter('sigma', SIGMA * unit.nanometer)
    force.addGlobalParameter('epsilon', EPSILON * unit.kilojoule_per_mole)
    force.addGlobalParameter('rcut', RCUT * unit.nanometer)
    force.setNonbondedMethod(mm.CustomNonbondedForce.CutoffPeriodic)
    force.setCutoffDistance(RCUT * unit.nanometer)
    force.setUseLongRangeCorrection(False)
    for _ in range(int(N)):
        force.addParticle([])
    system.addForce(force)
    return system, L

def make_topology(N):
    topology = app.Topology()
    chain = topology.addChain()
    residue = topology.addResidue('LJ', chain)
    element = app.Element.getByAtomicNumber(18)
    for _ in range(int(N)):
        topology.addAtom('Ar', element, residue)
    return topology

def get_positions(simulation):
    state = simulation.context.getState(getPositions=True)
    return np.asarray(state.getPositions(asNumpy=True).value_in_unit(unit.nanometer), dtype=float)

def lj_virial_pressure(positions, L, T):
    V = L ** 3
    rho = len(positions) / V
    virial = 0.0
    rcut2 = RCUT * RCUT
    for i in range(len(positions) - 1):
        delta = positions[i + 1:] - positions[i]
        delta -= L * np.rint(delta / L)
        r2 = np.sum(delta * delta, axis=1)
        mask = (r2 > 0.0) & (r2 < rcut2)
        if not np.any(mask):
            continue
        inv_r2 = (SIGMA * SIGMA) / r2[mask]
        inv_r6 = inv_r2 ** 3
        inv_r12 = inv_r6 ** 2
        virial += float(np.sum(24.0 * EPSILON * (2.0 * inv_r12 - inv_r6)))
    return float(rho * T + virial / (3.0 * V))

def sample_stats(values):
    values = [float(v) for v in values if math.isfinite(float(v))]
    if not values:
        return math.nan, math.nan
    if len(values) == 1:
        return float(values[0]), 0.0
    return float(mean(values)), float(stdev(values))

def append_row_csv(path, fieldnames, row):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    write_header = not path.exists() or path.stat().st_size == 0
    with path.open('a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        writer.writerow({key: row.get(key, '') for key in fieldnames})

def completed_run_ids(eos_path):
    path = Path(eos_path)
    if not path.exists() or path.stat().st_size == 0:
        return set()
    df = pd.read_csv(path)
    if 'status' in df.columns:
        df = df[df['status'] == 'ok']
    return set(df['run_id'].astype(str))

def save_failure(path, run_id, seed, run_seed, T, rho, exc):
    row = {'run_id': run_id, 'seed': int(seed), 'run_seed': int(run_seed), 'T_target': float(T), 'rho_target': float(rho), 'error_type': type(exc).__name__, 'error_message': str(exc), 'timestamp': datetime.now(timezone.utc).isoformat(timespec='seconds')}
    append_row_csv(path, FAILURE_FIELDS, row)

def create_cuda_platform():
    platform = mm.Platform.getPlatformByName('CUDA')
    properties = {'DeviceIndex': str(DEVICE_INDEX), 'Precision': str(PRECISION)}
    return platform, properties

def openmm_temperature(T):
    gas_r = unit.MOLAR_GAS_CONSTANT_R.value_in_unit(unit.kilojoule_per_mole / unit.kelvin)
    return (float(T) * EPSILON / gas_r) * unit.kelvin

def run_one_point(seed, T, rho):
    run_id = run_id_for(seed, T, rho)
    run_seed = deterministic_run_seed(seed, T, rho)
    system, L = create_lj_system(N, rho)
    target_T = openmm_temperature(T)
    integrator = mm.LangevinMiddleIntegrator(target_T, FRICTION / unit.picosecond, DT * unit.picoseconds)
    integrator.setRandomNumberSeed(int(run_seed))
    platform, properties = create_cuda_platform()
    simulation = app.Simulation(make_topology(N), system, integrator, platform, properties)
    simulation.context.setPositions(initialize_positions(N, L) * unit.nanometer)
    simulation.context.setVelocitiesToTemperature(target_T, int(run_seed))
    actual_platform = simulation.context.getPlatform().getName()
    if actual_platform != 'CUDA':
        raise RuntimeError('Expected CUDA platform, got ' + repr(actual_platform))
    start_time = time.perf_counter()
    if EQUIL_STEPS:
        simulation.step(int(EQUIL_STEPS))
    if PROD_STEPS % SAMPLE_INTERVAL != 0:
        raise ValueError('PROD_STEPS must be divisible by SAMPLE_INTERVAL')
    n_samples_expected = PROD_STEPS // SAMPLE_INTERVAL
    if n_samples_expected % N_BLOCKS != 0:
        raise ValueError('PROD_STEPS / SAMPLE_INTERVAL must be divisible by N_BLOCKS')
    samples_per_block = n_samples_expected // N_BLOCKS
    samples = []
    completed = 0
    while completed < PROD_STEPS:
        chunk = min(SAMPLE_INTERVAL, PROD_STEPS - completed)
        simulation.step(int(chunk))
        completed += int(chunk)
        state = simulation.context.getState(getEnergy=True)
        K_total = state.getKineticEnergy().value_in_unit(unit.kilojoule_per_mole)
        U_total = state.getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole)
        T_inst = 2.0 * K_total / (max(1, 3 * N - 3) * EPSILON)
        positions = get_positions(simulation)
        P_inst = lj_virial_pressure(positions, L, T_inst)
        U_per_particle = float(U_total) / N
        K_per_particle = float(K_total) / N
        samples.append({'step': int(completed), 'T': float(T_inst), 'P': float(P_inst), 'U': U_per_particle, 'K': K_per_particle, 'E': U_per_particle + K_per_particle})
        if not math.isfinite(T_inst) or T_inst > 10.0 * float(T):
            raise RuntimeError('Unstable temperature: T_inst=' + str(T_inst))
    wall_time_s = time.perf_counter() - start_time
    steps_per_second = (EQUIL_STEPS + PROD_STEPS) / wall_time_s if wall_time_s > 0 else math.nan
    result = {'run_id': run_id, 'seed': int(seed), 'run_seed': int(run_seed), 'T_target': float(T), 'rho_target': float(rho), 'N': int(N), 'box_length': float(L), 'sigma': float(SIGMA), 'epsilon': float(EPSILON), 'mass': float(MASS), 'rcut': float(RCUT), 'dt': float(DT), 'friction': float(FRICTION), 'equil_steps': int(EQUIL_STEPS), 'prod_steps': int(PROD_STEPS), 'sample_interval': int(SAMPLE_INTERVAL), 'n_samples': int(len(samples)), 'n_blocks': int(N_BLOCKS), 'platform': actual_platform, 'device_index': str(DEVICE_INDEX), 'precision': str(PRECISION), 'wall_time_s': float(wall_time_s), 'steps_per_second': float(steps_per_second), 'status': 'ok'}
    for key in ['T', 'P', 'U', 'K', 'E']:
        avg, std = sample_stats([x[key] for x in samples])
        result[key + '_mean'] = avg
        result[key + '_std'] = std
    return result


def cuda_smoke_check():
    old_N = globals()['N']
    try:
        globals()['N'] = 8
        result = run_one_point(seed=0, T=1.0, rho=0.05)
        print('CUDA smoke check ok:', result['platform'], 'steps/s:', format(result['steps_per_second'], '.1f'))
    finally:
        globals()['N'] = old_N

if USE_MULTIGPU_WORKERS:
    print('CUDA smoke check skipped in parent process; each GPU worker will create its own CUDA context.')
else:
    cuda_smoke_check()


In [ ]:
# Основной запуск заданных прогонов.
# Single-GPU пишет eos_targeted_points.csv.
# Multi-GPU пишет shard-файлы eos_targeted_points_gpu<N>.csv, чтобы процессы не ломали один CSV.

EOS_PATH = OUTPUT_DIR / 'eos_targeted_points.csv'
FAILURES_PATH = OUTPUT_DIR / 'failures_targeted_points.csv'


def completed_run_ids_many(paths):
    done = set()
    for path in paths:
        done.update(completed_run_ids(path))
    return done


def existing_targeted_eos_paths():
    return sorted(OUTPUT_DIR.glob('eos_targeted*.csv'))


def group_runs_by_seed(runs):
    grouped = {}
    for T, rho, seed in runs:
        grouped.setdefault(int(seed), []).append((float(T), float(rho)))
    return grouped


def run_worker_on_device(device_index, worker_runs, eos_path, failures_path, done_snapshot, worker_label):
    globals()['DEVICE_INDEX'] = str(device_index)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    local_done = set(done_snapshot) | completed_run_ids(eos_path)
    new_ok = 0
    failures = 0
    start = time.perf_counter()
    print('worker', worker_label, 'starting on CUDA DeviceIndex', DEVICE_INDEX, 'runs:', len(worker_runs), 'eos:', eos_path)
    try:
        cuda_smoke_check()
    except Exception as exc:
        print('worker', worker_label, 'CUDA smoke check failed:', type(exc).__name__, str(exc))
        raise

    for seed in sorted(group_runs_by_seed(worker_runs)):
        seed_points = group_runs_by_seed(worker_runs)[seed]
        print('worker', worker_label, 'seed', seed, 'points:', len(seed_points))
        for point_index, (T, rho) in enumerate(seed_points, start=1):
            run_id = run_id_for(seed, T, rho)
            run_seed = deterministic_run_seed(seed, T, rho)
            if run_id in local_done:
                continue
            try:
                point_start = time.perf_counter()
                result = run_one_point(seed, T, rho)
                append_row_csv(eos_path, EOS_FIELDS, result)
                local_done.add(run_id)
                new_ok += 1
                point_time = time.perf_counter() - point_start
                print('worker=' + str(worker_label) + ' device=' + str(DEVICE_INDEX) + ' seed=' + str(seed) + ' point=' + str(point_index) + '/' + str(len(seed_points)) + ' T=' + format(float(T), '.2f') + ' rho=' + format(float(rho), '.4f') + ' time=' + format(point_time, '.1f') + 's speed=' + format(result['steps_per_second'], '.1f') + ' steps/s file=' + str(eos_path))
            except Exception as exc:
                failures += 1
                save_failure(failures_path, run_id, seed, run_seed, T, rho, exc)
                print('FAILED worker=' + str(worker_label) + ' device=' + str(DEVICE_INDEX) + ' seed=' + str(seed) + ' T=' + format(float(T), '.2f') + ' rho=' + format(float(rho), '.4f') + ': ' + type(exc).__name__ + ': ' + str(exc))
                traceback.print_exc(limit=2)
    elapsed = time.perf_counter() - start
    print('worker', worker_label, 'finished; new ok:', new_ok, 'failures:', failures, 'wall time, s:', format(elapsed, '.1f'))
    return {'worker': worker_label, 'device_index': str(device_index), 'new_ok': int(new_ok), 'failures': int(failures), 'wall_time_s': float(elapsed)}


def worker_entry(device_index, worker_runs, eos_path_str, failures_path_str, done_snapshot, worker_label, queue):
    try:
        summary = run_worker_on_device(
            device_index=device_index,
            worker_runs=worker_runs,
            eos_path=Path(eos_path_str),
            failures_path=Path(failures_path_str),
            done_snapshot=done_snapshot,
            worker_label=worker_label,
        )
        queue.put(summary)
    except Exception as exc:
        queue.put({'worker': worker_label, 'device_index': str(device_index), 'new_ok': 0, 'failures': len(worker_runs), 'error_type': type(exc).__name__, 'error_message': str(exc)})
        raise


total_start = time.perf_counter()
all_existing_done = completed_run_ids_many(existing_targeted_eos_paths())
requested_not_done = [run for run in REQUESTED_RUNS if run_id_for(run[2], run[0], run[1]) not in all_existing_done]

print('Requested runs:', len(REQUESTED_RUNS))
print('Already completed in targeted files:', len(REQUESTED_RUNS) - len(requested_not_done))
print('Remaining runs:', len(requested_not_done))

if USE_MULTIGPU_WORKERS:
    print('Multi-GPU mode: one independent process per CUDA DeviceIndex:', DEVICE_INDICES)
    assignments = {str(device): [] for device in DEVICE_INDICES}
    for i, run in enumerate(requested_not_done):
        assignments[str(DEVICE_INDICES[i % len(DEVICE_INDICES)])].append(run)

    ctx = mp.get_context('fork')
    queue = ctx.Queue()
    processes = []
    for device_index in DEVICE_INDICES:
        runs = assignments[str(device_index)]
        if not runs:
            continue
        eos_path = OUTPUT_DIR / ('eos_targeted_points_gpu' + str(device_index) + '.csv')
        failures_path = OUTPUT_DIR / ('failures_targeted_points_gpu' + str(device_index) + '.csv')
        process = ctx.Process(
            target=worker_entry,
            args=(str(device_index), runs, str(eos_path), str(failures_path), all_existing_done, 'gpu' + str(device_index), queue),
        )
        process.start()
        processes.append(process)

    summaries = []
    for _ in processes:
        summaries.append(queue.get())
    for process in processes:
        process.join()
        if process.exitcode != 0:
            raise RuntimeError('GPU worker process failed with exit code ' + str(process.exitcode))
else:
    print('Single-GPU mode on CUDA DeviceIndex:', DEVICE_INDEX)
    summaries = [run_worker_on_device(
        device_index=DEVICE_INDEX,
        worker_runs=requested_not_done,
        eos_path=EOS_PATH,
        failures_path=FAILURES_PATH,
        done_snapshot=all_existing_done,
        worker_label='single',
    )]

print()
print('Targeted run finished.')
print('worker summaries:', summaries)
print('new successful points this run:', sum(int(x.get('new_ok', 0)) for x in summaries))
print('failures this run:', sum(int(x.get('failures', 0)) for x in summaries))
print('total wall time, s:', format(time.perf_counter() - total_start, '.1f'))


In [ ]:
# Финальный список файлов.

print('OUTPUT_DIR:', OUTPUT_DIR)
for path in sorted(OUTPUT_DIR.glob('eos_targeted*.csv')) + sorted(OUTPUT_DIR.glob('failures_targeted*.csv')):
    if path.exists():
        rows = max(0, sum(1 for _ in path.open('r', encoding='utf-8')) - 1)
        print(path.name + ': ' + str(rows) + ' data rows')
    else:
        print(path.name + ': not created')

successful_total = completed_run_ids_many(sorted(OUTPUT_DIR.glob('eos_targeted*.csv')))
print('successful targeted EOS points across all targeted files:', len(successful_total))
print('requested runs in this notebook:', len(REQUESTED_RUNS))
